In [1]:
import pandas as pd
import numpy as np
import random

def generate_stochastic_research_dataset(input_file):
    print("📖 Loading raw ASIN list...")
    # Load your raw data (e.g., AMAZON_FASHION.csv) to get the actual ASIN list
    raw_df = pd.read_csv(input_file)
    asin_list = raw_df['asin'].unique().tolist()
    
    final_rows = []
    total_asins = len(asin_list)
    print(f"🚀 Processing {total_asins} ASINs with Stochastic Zero-Sales Logic...")

    for index, asin in enumerate(asin_list):
        row = {'asin': asin}
        
        # 1. Determine how many months will have 0 sales (Randomly between 20 and 50)
        num_zero_months = random.randint(20, 50)
        
        # 2. Randomly select which month indices will be zero
        zero_month_indices = random.sample(range(1, 61), num_zero_months)
        
        # 3. Generate 60 months of sales
        for m in range(1, 61):
            if m in zero_month_indices:
                row[f'Month_{m}'] = 0
            else:
                # Logic: Real review count * random multiplier (4 to 6)
                # If you don't have per-month reviews, we simulate a 'Review Event' (1-10)
                sim_reviews = random.randint(1, 10) 
                multiplier = random.uniform(4, 6)
                row[f'Month_{m}'] = int(sim_reviews * multiplier)

        # 4. Randomized Current Stock based on the most recent demand (Month 60)
        # Using a logic to create varied 'Velocity' ratios for the Triage system
        last_m_sales = row['Month_60']
        # If last month was 0, we use a baseline for stock generation
        stock_base = last_m_sales if last_m_sales > 0 else 10
        row['current_stock'] = random.randint(int(stock_base * 0.2), int(stock_base * 3))
        
        final_rows.append(row)
        
        if (index + 1) % 20000 == 0:
            print(f"✅ Processed {index + 1} / {total_asins} ASINs...")

    # Final Save
    print("💾 Finalizing DataFrame...")
    final_df = pd.DataFrame(final_rows)
    final_df.to_csv('final_fashion_dataset(random).csv', index=False)
    
    print("-" * 50)
    print(f"📂 SUCCESS: 'final_fashion_dataset(random).csv' generated.")
    print(f"📊 Dataset Density: {((60 - 35)/60)*100:.1f}% Avg. Non-Zero Activity")
    print("-" * 50)
    
    return final_df

# Run in your environment
df = generate_stochastic_research_dataset("AMAZON_FASHION.csv")

📖 Loading raw ASIN list...
🚀 Processing 186189 ASINs with Stochastic Zero-Sales Logic...
✅ Processed 20000 / 186189 ASINs...
✅ Processed 40000 / 186189 ASINs...
✅ Processed 60000 / 186189 ASINs...
✅ Processed 80000 / 186189 ASINs...
✅ Processed 100000 / 186189 ASINs...
✅ Processed 120000 / 186189 ASINs...
✅ Processed 140000 / 186189 ASINs...
✅ Processed 160000 / 186189 ASINs...
✅ Processed 180000 / 186189 ASINs...
💾 Finalizing DataFrame...
--------------------------------------------------
📂 SUCCESS: 'final_fashion_dataset(random).csv' generated.
📊 Dataset Density: 41.7% Avg. Non-Zero Activity
--------------------------------------------------


In [2]:
import pandas as pd
import numpy as np
import random

def generate_intermittent_dataset(input_file):
    print("📖 Loading AMAZON_FASHION.csv...")
    df = pd.read_csv(input_file)
    
    # 1. Map real review timestamps
    df['reviewTime'] = pd.to_datetime(df['reviewTime'])
    end_date = df['reviewTime'].max()
    start_date = end_date - pd.DateOffset(months=59)
    df_filtered = df[df['reviewTime'] >= start_date].copy()
    
    # 2. Count real reviews per ASIN per Month
    df_filtered['year_month'] = df_filtered['reviewTime'].dt.to_period('M')
    review_counts = df_filtered.groupby(['asin', 'year_month']).size().reset_index(name='rc')
    
    # 3. Pivot to 60-month format
    all_months = sorted(df_filtered['year_month'].unique())
    month_map = {month: f'Month_{i+1}' for i, month in enumerate(all_months)}
    review_counts['m_label'] = review_counts['year_month'].map(month_map)
    pivot_df = review_counts.pivot(index='asin', columns='m_label', values='rc').fillna(0)
    
    # Ensure all 60 columns exist
    for i in range(1, 61):
        if f'Month_{i}' not in pivot_df.columns: pivot_df[f'Month_{i}'] = 0
            
    final_rows = []
    print(f"🧪 Applying Stochastic Intermittency (20-50 Zero-Months)...")

    for asin, row in pivot_df.iterrows():
        new_row = {'asin': asin}
        
        # REQUIREMENT: Force 20 to 50 months to be ZERO sales
        num_zeros = random.randint(0, 25)
        zero_indices = random.sample(range(1, 61), num_zeros)
        
        for m in range(1, 61):
            if m in zero_indices:
                new_row[f'Month_{m}'] = 0
            else:
                # REQUIREMENT: Actual reviews * random multiplier (4 to 6)
                actual_reviews = row[f'Month_{m}']
                # If real reviews are 0 but this isn't a 'forced zero' month, 
                # we use a baseline of 1 to ensure some activity
                base = max(1, actual_reviews) 
                new_row[f'Month_{m}'] = int(base * random.uniform(4, 6))
        
        final_rows.append(new_row)

    # 4. Save Final Dataset (No current_stock as requested)
    pd.DataFrame(final_rows).to_csv('final_fashion_dataset(r1).csv', index=False)
    print("📂 SUCCESS: 'final_fashion_dataset(r1).csv' created with high sparsity.")
    return len(final_rows)

df_count = generate_intermittent_dataset("AMAZON_FASHION.csv")

📖 Loading AMAZON_FASHION.csv...
🧪 Applying Stochastic Intermittency (20-50 Zero-Months)...
📂 SUCCESS: 'final_fashion_dataset(r1).csv' created with high sparsity.
